## 🛠️ Environment Setup

In [ ]:
import os
import time
import shutil
import subprocess
from pathlib import Path

from utils.init_utils import define_hotspot
from utils.boltz_2_design_utils import generate_boltz_input, extract_Boltz_metrics
from utils.screen_utils import analyze_and_filter_predictions, extract_colabfold_metrics
from utils.convert_utils import format_time, convert_unk_to_ala, save_metrics_to_json
from utils.rosetta_utils import run_PyRosetta_interface_analysis

start_time = time.time()

## 🗺 Dir Initialization

In [ ]:
# ===== Modify based on your actual situation =====
PROJECT_NAME = 'NK3R_Linear_Peptide_Binder_L12'

BASE_DIR = Path('./demo_display')
PROJECT_DIR = Path(BASE_DIR, PROJECT_NAME)
REFERENCE_PDB_BASE_DIR = Path(PROJECT_DIR, 'reference_complex_pdbs')
ANALYSIS_RESULTS_DIR = Path(PROJECT_DIR, 'analysis_results')

BASE_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_PDB_BASE_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 🔴 Define Hotspot
<h4 style="color:purple;">🤔 Now, put your reference PDB files into the ANALYSIS_RESULTS_DIR!</h4>

In [ ]:
# ===== Modify based on your actual situation =====
KNOWN_COMPLEX_PDBS = [
    Path(REFERENCE_PDB_BASE_DIR, 'NK3R_NKB.pdb'),
    Path(REFERENCE_PDB_BASE_DIR,'NK3R_SP.pdb'),
    Path(REFERENCE_PDB_BASE_DIR,'NK3R_Senktide.pdb')
]  

RECEPTOR_CHAIN_IDS_for_hotspot_define = ['B']  # PDB_file
PEPTIDE_CHAIN_ID_for_hotspot_define = 'A'  # PDB_file

RECEPTOR_CHAIN_IDS = ['B']  # for design, if multichain, you should enter in order, such as ['B', 'C', 'D', ...].
PEPTIDE_CHAIN_ID = 'A'  # for design, peptide chain always be 'A'.

CONTACT_DISTANCE = 5.0
OUTPUT_HOTSPOT_FILE = Path(ANALYSIS_RESULTS_DIR, 'hotspot_residues.json')

In [ ]:
define_hotspot(known_complex_pdbs=KNOWN_COMPLEX_PDBS, 
               receptor_chain_ids=RECEPTOR_CHAIN_IDS_for_hotspot_define, peptide_chain_ids=PEPTIDE_CHAIN_ID_for_hotspot_define,
               contact_distance=CONTACT_DISTANCE, output_hotspot_file=OUTPUT_HOTSPOT_FILE)

## ❓Boltz-2 Xpep hallucination design

In [ ]:
# ===== Modify based on your actual situation =====
CONDA_ENVIRONMENT_FOR_Boltz = Path('/home/qiuyk/miniconda3/envs/Boltz/bin')
LOOP_DIR = Path(PROJECT_DIR, 'Xpep_Boltz-2_Hallucination_Cycle')

# Different chains are separated by ':'
RECEPTOR_PROTEIN_SEQ = 'MATLPAAETWIDGGGGVGADAVNLTASLAAGAATGAVETGWLQLLDQAGNLSSSPSALGLPVASPAPSQPWANLTNQFVQPSWRIALWSLAYGVVVAVAVLGNLIVIWIILAHKRMRTVTNYFLVNLAFSDASMAAFNTLVNFIYALHSEWYFGANYCRFQNFFPITAVFASIYSMTAIAVDRYMAIIDPLKPRLSATATKIVIGSIWILAFLLAFPQCLYSKTKVMPGRTLCFVQWPEGPKQHFTYHIIVIILVYCFPLLIMGITYTIVGITLWGGEIPGDTCDKYHEQLKAKRKVVKMMIIVVMTFAICWLPYHIYFILTAIYQQLNRWKYIQQVYLASFWLAMSSTMYNPIIYCCLNKRFRAGFKRAFRWCPFIKVSSYDELELKTTRFHPNRQSSMYTVTRMESMTVVFDPNDADTTRSSRKKRATPRDPSFNGCSRRNSKSASATSSFISSPYTSVDEYS'
X_length = 12
IS_CYCLIC_PEPTIDE = False  # Linear peptide bidner OR Cyclic peptide binder

# Suceess Threshold
CLASH_DISTANCE_THRESHOLD = 2.5
HOTSPOT_CONTACT_DISTANCE = 5.0
MAX_CLASHES_ALLOWED = 3
HOTSPOT_COVERAGE_THRESHOLD_Boltz = 35.0 # (35%, ensure that the peptide is at the intended target location, need to modify acrooding to X_length)
ALANINE_PERCENTAGE_THRESHOLD = 25.0 # (25%, avoid generating sequences that appear stable but are actually unreasonable)

In [ ]:
# Boltz-2 Sripts, Modify Boltz-2's setup options based on your actual situation
def run_boltz_prediction(input_yaml, output_dir, conda_bin_dir):
    print(f"--- Starting Boltz-2 Prediction ---")
    print(f"Input: {input_yaml}")
    print(f"Output: {output_dir}")
    boltz_executable = conda_bin_dir / 'boltz'
    cmd = [
        str(boltz_executable), "predict", str(input_yaml),
        "--out_dir", str(output_dir),
        "--use_msa_server", 
        "--use_potentials",
        "--diffusion_samples", "1",
        "--output_format", "pdb",
        "--max_parallel_samples", "1"
    ]
    print(f"Running command: {' '.join(cmd)}\n")
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            encoding='utf-8',
            check=False
        )
        if result.returncode == 0:
            print("--- Boltz-2 prediction finished successfully. ---")
            return True
        else:
            print(f"--- Boltz-2 prediction failed (Return Code: {result.returncode}). ---")
            print("--- Boltz-2 Output (on Failure) ---")
            print(result.stdout)
            if result.stderr:
                print("--- Boltz-2 Stderr (on Failure) ---")
                print(result.stderr)
            return False
    except Exception as e:
        print(f"An unexpected error occurred while running Boltz-2: {e}")
        return False

## ♻️ Iterative Design Cycle

In [ ]:
# for REAPS
CONDA_ENVIRONMENT_FOR_REAPS = Path('/home/qiuyk/miniconda3/envs/REAPS/bin/python')
REAPS_SCRIPT_PATH = Path("../REAPS/inference.py")
REAPS_CHECKPOINT_PATH = Path("../checkpoints/RAPID_n0.02_pepFT.ckpt")
REAPS_CONFIG_PATH = Path("../configs/model/REAPS.yaml")
REAPS_NUM_SAMPLES = 1 
REAPS_TEMPERATURE = 0.1
REAPS_SEED = 3407

# for ColabFold-AF2_multimer or HighFold-multimer
CONDA_ENVIRONMENT_FOR_ColabFold_AF2_multimer = Path('/home/qiuyk/code/LocalColabFold/localcolabfold/colabfold-conda/bin/colabfold_batch')
# CONDA_ENVIRONMENT_FOR_ColabFold_AF2_multimer = Path('/home/qiuyk/code/LocalColabFold/localcolabfold/highfold-conda/bin/colabfold_batch')

TARGET_CANDIDATES_TO_GENERATE = 1 # num of design
MAX_ATTEMPTS_PER_TEMPLATE = 1 # max attempts for Cycle-0
REFINE_CYCLES_PER_DESIGN = 2 # num of refine cycle per design

Xpep_HALLUCINATION_DESIGN_YAML = Path(LOOP_DIR, 'Xpep_hallucination_sequence.yaml')

generate_boltz_input(
    receptor_seq_str=RECEPTOR_PROTEIN_SEQ, 
    peptide_length=X_length, 
    output_file_path=Xpep_HALLUCINATION_DESIGN_YAML,
    is_cyclic=IS_CYCLIC_PEPTIDE
)

In [ ]:
def generate_and_filter_new_template(cycle_0_dir, design_attempt_id):
    print(f"--- Generating and Filtering New Template (Attempt #{design_attempt_id}) ---")
    template_gen_dir = cycle_0_dir
    attempt = 0
    while attempt < MAX_ATTEMPTS_PER_TEMPLATE:
        attempt += 1
        print(f"===== Template Generation Attempt {attempt} / {MAX_ATTEMPTS_PER_TEMPLATE} =====")
        if attempt > 1: 
            print(f"Cleaning up previous failed attempt in {template_gen_dir}...")
            try:
                shutil.rmtree(template_gen_dir)
            except OSError as e:
                print(f"Error cleaning directory: {e}. Stopping.")
                return None
            template_gen_dir.mkdir(parents=True, exist_ok=True)
        prediction_ok = run_boltz_prediction(
            input_yaml=Xpep_HALLUCINATION_DESIGN_YAML,
            output_dir=template_gen_dir,
            conda_bin_dir=CONDA_ENVIRONMENT_FOR_Boltz
        )
        if not prediction_ok:
            print("Prediction failed. Skipping analysis and retrying.")
            continue 
        passed_pdb_path = analyze_and_filter_predictions(
            base_dir=template_gen_dir,
            hotspot_file=OUTPUT_HOTSPOT_FILE,
            receptor_chain_ids=RECEPTOR_CHAIN_IDS,
            peptide_chain_id=PEPTIDE_CHAIN_ID,
            clash_threshold=CLASH_DISTANCE_THRESHOLD,
            max_clashes=MAX_CLASHES_ALLOWED,
            hotspot_contact_dist=HOTSPOT_CONTACT_DISTANCE,
            hotspot_coverage_thresh=HOTSPOT_COVERAGE_THRESHOLD_Boltz,
            yaml_path=Xpep_HALLUCINATION_DESIGN_YAML
        )
        if passed_pdb_path:
            print(f"===== SUCCESS! New Template Found. ======")
            print(f"Template for Attempt #{design_attempt_id} found on try {attempt}: {passed_pdb_path.name}")
            return passed_pdb_path
        else:
            if attempt < MAX_ATTEMPTS_PER_TEMPLATE:
                print(f"Template generation attempt {attempt} failed filters. Retrying...")
            else:
                print(f"Maximum attempts ({MAX_ATTEMPTS_PER_TEMPLATE}) reached for generating template #{design_attempt_id}. FAILED.")
                return None
    print(f"\n====== Failed to generate template #{design_attempt_id}. ======")
    return None

In [ ]:
# REAPS peptide sequences design
def run_REAPS(temp_pdb_path, specific_fasta_output_dir):
    run_REAPS_cmd = [
            str(CONDA_ENVIRONMENT_FOR_REAPS.resolve()),
            str(REAPS_SCRIPT_PATH.resolve()),
            "--pdb_file", str(temp_pdb_path.resolve()),
            "--peptide_chain_id", PEPTIDE_CHAIN_ID,
            "--checkpoint_path", REAPS_CHECKPOINT_PATH ,
            "--model_config_path", REAPS_CONFIG_PATH,
            "--fasta_output_path", str(specific_fasta_output_dir.resolve()),
            "--num_samples", str(REAPS_NUM_SAMPLES),
            "--temperature", str(REAPS_TEMPERATURE),
            "--seed", str(REAPS_SEED),
            "--mode", 'cyclic' if (IS_CYCLIC_PEPTIDE and 'cyclic' in REAPS_CHECKPOINT_PATH.stem) else 'linear',
        ]
    try:
        result = subprocess.run(run_REAPS_cmd, check=True, text=True, capture_output=True)
        print("  --- REAPS STDOUT ---")
        print(result.stdout if result.stdout else " (No stdout)")
        # print("  --- REAPS STDERR ---")
        # print(result.stderr if result.stderr else " (No stderr)")
        print("  ✓ REAPS sample complete.")
        seqs_dir = Path(specific_fasta_output_dir, 'seqs')
        fasta_files = list(seqs_dir.glob("*.fa"))
        target_fasta_file = fasta_files[0]
        sequence_full = ""
        found_sample_header = False
        try:
            with open(target_fasta_file, 'r') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    if line.startswith(">sample_"):
                        found_sample_header = True
                        continue
                    if found_sample_header:
                        sequence_full = line
                        break
        except Exception as e:
            print(f"  - !!!!! REAPS Error: Failed to read FASTA file {target_fasta_file}: {e} !!!!!")
            return None
        if not sequence_full:
            if found_sample_header:
                print("  - !!!!! REAPS Error: Found '>sample_' header, but sequence line is empty. !!!!!")
            else:
                print("  - !!!!! REAPS Error: Could not find a '>sample_' header in the FASTA file. !!!!!")
            return None
        if ":" in sequence_full:
            sequence = sequence_full.split(':')[0]
        else:
            sequence = sequence_full
        print(f"  ✓ REAPS Designed Sequence: {sequence[:X_length]}\n")
        return sequence
    except subprocess.CalledProcessError as e:
        print(f"  - !!!!! REAPS Run Failed (Exit Code: {e.returncode}) !!!!!")
        print("✓ --- REAPS STDOUT ---")
        print(e.stdout)
        # print("✓ --- REAPS STDERR ---")
        # print(e.stderr)
        return None

In [ ]:
def run_boltz_refinement_and_filter(new_peptide_seq, receptor_seq, cycle_dir, attempt_id, cycle_num):
    print(f"--- Running Boltz-2 refinement for cycle {cycle_num}... ---")

    refinement_yaml = cycle_dir / f"design_{attempt_id}_cycle{cycle_num}.yaml"

    receptor_sequences = receptor_seq.split(':')

    yaml_lines = [
        "version: 1",
        "sequences:",
        "  - protein:",
        f"      id: {PEPTIDE_CHAIN_ID}",
        f"      sequence: {new_peptide_seq.strip()}",
    ]

    if IS_CYCLIC_PEPTIDE:
        yaml_lines.append("      cyclic: true")

    yaml_lines.append("      msa: empty")

    for i, seq in enumerate(receptor_sequences):
        chain_id = chr(ord('B') + i)
        yaml_lines.extend([
            "  - protein:",
            f"      id: {chain_id}",
            f"      sequence: {seq.strip()}",
            "      msa: empty",
        ])

    try:
        with open(refinement_yaml, "w") as f:
            f.write("\n".join(yaml_lines) + "\n")
    except IOError as e:
        print(f"  - !!!!! YAML Write Error: {e} !!!!!")
        return None

    prediction_ok = run_boltz_prediction(
        input_yaml=refinement_yaml,
        output_dir=cycle_dir,
        conda_bin_dir=CONDA_ENVIRONMENT_FOR_Boltz
    )

    if not prediction_ok:
        print("  - Boltz-2 refinement prediction failed.")
        return None

    passed_pdb_path = analyze_and_filter_predictions(
        base_dir=cycle_dir,
        hotspot_file=OUTPUT_HOTSPOT_FILE,
        receptor_chain_ids=RECEPTOR_CHAIN_IDS,
        peptide_chain_id=PEPTIDE_CHAIN_ID,
        clash_threshold=CLASH_DISTANCE_THRESHOLD,
        max_clashes=MAX_CLASHES_ALLOWED,
        hotspot_contact_dist=HOTSPOT_CONTACT_DISTANCE,
        hotspot_coverage_thresh=HOTSPOT_COVERAGE_THRESHOLD_Boltz,
        yaml_path=refinement_yaml
    )

    return passed_pdb_path

In [ ]:
print("\n=====================================================================")
print(f"      Iterative Generation (Generating {TARGET_CANDIDATES_TO_GENERATE} Candidates)")
print("=====================================================================")
promising_candidates = []
for design_attempt_id in range(1, TARGET_CANDIDATES_TO_GENERATE + 1):
    print(f"\n================= Generate Candidate {design_attempt_id} / {TARGET_CANDIDATES_TO_GENERATE} ================")
    current_design_dir = LOOP_DIR / f"design_attempt_{design_attempt_id}"
    current_design_dir.mkdir(parents=True, exist_ok=True)
    best_loop_iptm = -1.0
    best_loop_pdb = None
    best_loop_seq = None
    pdb_for_next_REAPS = None
    for cycle in range(REFINE_CYCLES_PER_DESIGN + 1):
        cycle_dir = current_design_dir / f"cycle_{cycle}"
        cycle_dir.mkdir(parents=True, exist_ok=True)
        current_pdb_path = None
        current_sequence = None
        # Cycle-0: init full-X hallucination
        if cycle == 0:
            print("\n★ Cycle 0: Hallucination (Boltz-2) ")
            pdb_cycle_0 = generate_and_filter_new_template(
                cycle_0_dir=cycle_dir,
                design_attempt_id=design_attempt_id
            )
            if pdb_cycle_0 is None:
                print("  - Cycle 0 Hallucination FAILED. Skipping this attempt.")
                break
            print(f"  ✓ Cycle 0 Template found: {pdb_cycle_0.name}")
            pdb_for_next_REAPS = pdb_cycle_0
            current_pdb_path = pdb_cycle_0
            ala_pdb_path = cycle_dir / f"design_{design_attempt_id}_cycle{cycle}_ALA.pdb"
            convert_unk_to_ala(pdb_for_next_REAPS, ala_pdb_path, PEPTIDE_CHAIN_ID)
            current_sequence = "A" * X_length
            pdb_for_next_REAPS = ala_pdb_path
        # Cycle-1+: refinement iteration
        else:
            print(f"\n★ Cycle {cycle}: Refinement (REAPS -> Boltz-2) ")
            REAPS_fasta_dir = cycle_dir / "REAPS_output"
            new_sequence = run_REAPS(pdb_for_next_REAPS, REAPS_fasta_dir)
            if new_sequence is None:
                print("  - REAPS design FAILED. Skipping this attempt.")
                break
            pdb_cycle_n = run_boltz_refinement_and_filter(
                new_peptide_seq=new_sequence,
                receptor_seq=RECEPTOR_PROTEIN_SEQ,
                cycle_dir=cycle_dir,
                attempt_id=design_attempt_id,
                cycle_num=cycle
            )
            if pdb_cycle_n is not None:
                print(f"  ✓ Boltz-2 refinement (Cycle {cycle}) PASSED filters.")
                pdb_for_next_REAPS = pdb_cycle_n
                current_pdb_path = pdb_cycle_n
                current_sequence = new_sequence
            else:
                print(f"  - Boltz-2 refinement (Cycle {cycle}) FAILED filters. Ending internal loop.")
                break
        if current_pdb_path is None:
            continue
        boltz_metrics = extract_Boltz_metrics(current_pdb_path.parent)
        if boltz_metrics is None:
            print("  - Warning: Could not extract Boltz metrics for this cycle. Skipping 'best' check.")
            continue
        current_iptm = boltz_metrics.get("iptm", -1.0)
        alanine_percentage = 0.0
        if current_sequence and current_sequence != ("X" * X_length):
            alanine_count = current_sequence.count('A')
            alanine_percentage = (alanine_count / len(current_sequence)) * 100
        print(f"  - Cycle {cycle} Stats: ipTM = {current_iptm:.4f}, Ala% = {alanine_percentage:.2f}")
        is_better = current_iptm > best_loop_iptm
        is_not_all_ala = alanine_percentage <= ALANINE_PERCENTAGE_THRESHOLD
        if is_better and is_not_all_ala:
            print(f"  - ★★★ NEW BEST FOUND ★★★ (ipTM {current_iptm:.4f} > {best_loop_iptm:.4f})")
            best_loop_iptm = current_iptm
            best_loop_pdb = current_pdb_path
            best_loop_seq = current_sequence
        elif is_better and not is_not_all_ala:
            print(f"  - (Note: ipTM is better, but Ala% ({alanine_percentage:.2f}) is too high. Discarding.)")
        else:
            print(f"  - (Note: ipTM {current_iptm:.4f} is not better than {best_loop_iptm:.4f}. Discarding.)")
    if best_loop_seq and best_loop_seq != ("X" * X_length):
        print(f"  ✓ Candidate {design_attempt_id} generated successfully.")
        print(f"    - Best sequence (from best ipTM): {best_loop_seq[:X_length]}")
        print(f"    - Best PDB (from best ipTM): {best_loop_pdb.name}")
        promising_candidates.append({
            "id": f"design_attempt_{design_attempt_id}",
            "peptide_seq": best_loop_seq,
            "pdb_path": best_loop_pdb
        })
    else:
        print(f"  - Candidate {design_attempt_id} internal loop failed, no valid design was produced.")
print(f"\n\n===== Generated {len(promising_candidates)} promising candidates in total. =====")
if promising_candidates:
    print("\n--- Final List of Promising Candidates ---")
    for i, candidate in enumerate(promising_candidates, 1):
        print(f"  {i}. ID: {candidate['id']}")
        print(f"     Sequence: {candidate['peptide_seq']}")
        print(f"     PDB File: {candidate['pdb_path'].name}")

## 🖊 External validation

In [ ]:
# !!!!! Modify according to the reference structures !!!!!

pLDDT_THRESHOLD = 60.0
pTM_THRESHOLD = 0.5
ipTM_THRESHOLD = 0.8
ipAE_THRESHOLD = 25.0   # (Lower is better)
dG_THRESHOLD = -50.0    # (Lower is better)
HOTSPOT_COVERAGE_THRESHOLD_SCREEN = 85.0 # (85%, ensure that the peptide is at the intended target location, need to modify acrooding to X_length)

In [ ]:
# ColabFold predict structure
def run_ColabFold_AF2_multimer(input_fasta_dir, batch_output_dir):
    run_ColabFold_AF2_multimer_cmd = [
        str(CONDA_ENVIRONMENT_FOR_ColabFold_AF2_multimer),         
        str(input_fasta_dir), 
        str(batch_output_dir),
        "--num-models", "1",
        "--model-type", "alphafold2_multimer_v3",
        "--num-recycle", "3",
        "--amber",
        "--num-relax", "1",
        "--use-gpu-relax"
    ]
    env = os.environ.copy()
    env['MPLBACKEND'] = 'Agg'
    try:
        result = subprocess.run(
            run_ColabFold_AF2_multimer_cmd, 
            check=True, 
            text=True, 
            capture_output=True, 
            encoding='utf-8',
            env=env
        )
        print("  ✓ ColabFold prediction complete.")
        return
    except subprocess.CalledProcessError as e:
        print(f"  - !!!!! ColabFold Run Failed (Exit Code: {e.returncode}) !!!!!")
        print("  --- ColabFold STDOUT ---")
        print(e.stdout) 
        print("  --- ColabFold STDERR ---")
        print(e.stderr) 
        return

In [ ]:
# Check final metrics and thresholds
def check_metrics(colabfold_metrics: dict, rosetta_metrics: dict) -> bool:
    """
    Checks if a predicted structure meets all hard thresholds for success.
    Args:
        colabfold_metrics (dict): Metrics extracted from ColabFold (pLDDT, pTM, ipTM, mean_interface_pAE, etc.).
        rosetta_metrics (dict): Metrics extracted from PyRosetta (dG_separated, hotspot_coverage, etc.).
    Returns:
        bool: True if ALL conditions are met, False otherwise.
    """
    # Check for critical errors first
    if 'error' in rosetta_metrics:
        print(f"  CRITICAL FAILURE: Rosetta analysis error: {rosetta_metrics['error']}")
        return False
    if colabfold_metrics is None or not colabfold_metrics:
        print(f"  CRITICAL FAILURE: ColabFold metrics extraction failed.")
        return False
        
    # ColabFold Metrics
    plddt_score = colabfold_metrics.get('pLDDT', -1.0)
    ptm_score = colabfold_metrics.get('pTM', -1.0)
    iptm_score = colabfold_metrics.get('ipTM', -1.0)
    # ipAE corresponds to mean_pae_interface
    ipae_score = colabfold_metrics.get('mean_interface_pAE', 999.0) 
    
    # Rosetta Metrics
    dG_separated = rosetta_metrics.get('dG_separated', 999.0)
    hotspot_coverage = rosetta_metrics.get('receptor_hotspot_coverage_percent', 0.0)

    plddt_pass = plddt_score > pLDDT_THRESHOLD
    print(f"  pLDDT Check ({plddt_score:.4f} > {pLDDT_THRESHOLD}): {'PASS' if plddt_pass else 'FAIL'}")
    ptm_pass = ptm_score > pTM_THRESHOLD
    print(f"  pTM Check ({ptm_score:.4f} > {pTM_THRESHOLD}): {'PASS' if ptm_pass else 'FAIL'}")
    iptm_pass = iptm_score > ipTM_THRESHOLD
    print(f"  ipTM Check ({iptm_score:.4f} > {ipTM_THRESHOLD}): {'PASS' if iptm_pass else 'FAIL'}")
    ipae_pass = ipae_score < ipAE_THRESHOLD
    print(f"  ipAE Check ({ipae_score:.4f} < {ipAE_THRESHOLD:.4f}): {'PASS' if ipae_pass else 'FAIL'}")
    dg_pass = dG_separated < dG_THRESHOLD
    print(f"  Rosetta dG Check ({dG_separated:.4f} < {dG_THRESHOLD:.4f}): {'PASS' if dg_pass else 'FAIL'}")
    hotspot_pass = hotspot_coverage > HOTSPOT_COVERAGE_THRESHOLD_SCREEN
    print(f"  Hotspot Coverage Check ({hotspot_coverage:.2f}% > {HOTSPOT_COVERAGE_THRESHOLD_SCREEN:.1f}%): {'PASS' if hotspot_pass else 'FAIL'}")

    overall_success = plddt_pass and ptm_pass and iptm_pass and ipae_pass and dg_pass and hotspot_pass

    if overall_success:
        print("\nOVERALL SUCCESS: All metrics passed!")
    else:
        print("\nOVERALL FAILURE: One or more conditions were NOT met.")

    return overall_success

In [ ]:
print("\n=====================================================================")
print(f"         External Validation (AF2/HighFold + Rosetta)")
print(f"         Will validate {len(promising_candidates)} candidates...")
print("=====================================================================")

successful_designs_found = 0
validated_designs_log = []
SUCCESS_DIR = Path(PROJECT_DIR, 'Xpep_final_validated_designs') 
SUCCESS_DIR.mkdir(parents=True, exist_ok=True)

if not promising_candidates:
    print("Warning: No promising candidates were generated.")

for i, candidate in enumerate(promising_candidates, 1):
    
    candidate_id = candidate['id']
    candidate_seq = candidate['peptide_seq']
    print(f"\n=== Validating {candidate_id} ({i} / {len(promising_candidates)}) ===")
    validation_dir = LOOP_DIR / candidate_id / "final_validation"
    validation_dir.mkdir(parents=True, exist_ok=True)
    af2_input_dir = validation_dir / "af2_input"
    af2_input_dir.mkdir(parents=True, exist_ok=True)
    af2_fasta_path = af2_input_dir / f"{candidate_id}_final.fasta"
    try:
        all_chains = [candidate_seq.strip()]
        receptor_sequences = RECEPTOR_PROTEIN_SEQ.split(':')
        all_chains.extend([seq.strip() for seq in receptor_sequences])
        final_sequence_string = ":".join(all_chains)
        with open(af2_fasta_path, 'w') as f:
            f.write(f">{candidate_id}_ColabFold_AF2_multimer\n") 
            f.write(f"{final_sequence_string}\n")
    except IOError as e:
        print(f"  - !!!!! FASTA Write Error: {e}. Skipping this candidate. !!!!!")
        continue
    af2_output_dir = validation_dir / "af2_output"
    run_ColabFold_AF2_multimer(af2_input_dir, af2_output_dir)
    print("  Running Rosetta validation...")
    af2_metrics = extract_colabfold_metrics(af2_output_dir, X_length)
    rosetta_metrics = run_PyRosetta_interface_analysis(
        af2_output_dir, PEPTIDE_CHAIN_ID, RECEPTOR_CHAIN_IDS, OUTPUT_HOTSPOT_FILE, do_relax=True
    )
    print("  Checking final metrics against *STRICT* thresholds...")
    is_success = check_metrics(af2_metrics, rosetta_metrics)
    if is_success:
        successful_designs_found += 1
        print(f"  ======= 🎉 VALIDATION SUCCESS! (Total {successful_designs_found}) =======")
        final_pdb_path = af2_metrics.get('predicted_pdb_path')
        if final_pdb_path and Path(final_pdb_path).exists():
            success_name = f"{candidate_id}_final_validated"
            shutil.copy(final_pdb_path, SUCCESS_DIR / f"{success_name}.pdb")
            save_metrics_to_json(af2_metrics, rosetta_metrics, SUCCESS_DIR, success_name)
            validated_designs_log.append(candidate_id)
        else:
            print("  - Warning: Could not find AF2 PDB path to copy.")
    else:
        print(f"  - 😿 {candidate_id} FAILED: Did not meet strict AF2/HighFold + Rosetta metrics.")
print(f"Validated {len(promising_candidates)} candidates, found {successful_designs_found} successful designs.")
print(f"Final successful designs are saved in: {SUCCESS_DIR.resolve()}")
print(f"Successful Design IDs: {validated_designs_log}")

## ⏰ Time Report

In [ ]:
# Report time spent
end_time = time.time()
elapsed_time = end_time - start_time
print("\n====== Workflow Complete. ======")
print(f"Total time spent for {PROJECT_NAME} design: {format_time(elapsed_time)}.")

## 🎇 Over!!!